# Transfer Learning with Tensorflow : Fine Tuning 

In [1]:
# Check if we're using a GPU 
!nvidia-smi

Wed Feb 25 20:24:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

--2026-02-25 20:24:15--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0.001s  

2026-02-25 20:24:15 (17.7 MB/s) - ‘helper_functions.py’ saved [10246/10246]



## Get some data 

link : https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

In [3]:
# Import helper functions 
from helper_functions import create_tensorboard_callback , plot_loss_curves , unzip_data , walk_through_dir

In [4]:
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

unzip_data("10_food_classes_10_percent.zip")

--2026-02-25 20:24:24--  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.135.207, 192.178.163.207, 74.125.142.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.135.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 168546183 (161M) [application/zip]
Saving to: ‘10_food_classes_10_percent.zip’

10_food_classes_10_ 100%[===================>] 160.74M  41.9MB/s    in 3.8s    

2026-02-25 20:24:28 (41.9 MB/s) - ‘10_food_classes_10_percent.zip’ saved [168546183/168546183]



In [5]:
# Check out how many images and subdirectories are in our dataset 
walk_through_dir("10_food_classes_10_percent")

There are 2 directories and 0 images in '10_food_classes_10_percent'.
There are 10 directories and 0 images in '10_food_classes_10_percent/train'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/chicken_wings'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/sushi'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/ramen'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/grilled_salmon'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/steak'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/hamburger'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/fried_rice'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/pizza'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/ice_cream'.
There are 0 directories and 75 images in '10_food_classes_10_percent/train/c

In [6]:
# Create training and test directories paths 
train_dir = "10_food_classes_10_percent/train/"
test_dir = "10_food_classes_10_percent/test/"

In [7]:
import tensorflow as tf

IMG_SIZE = (244, 244)
BATCH_DATA = 32

train_data_10_percent = tf.keras.preprocessing.image_dataset_from_directory(directory=train_dir,
                                                                            image_size=IMG_SIZE,
                                                                            label_mode="categorical",
                                                                            batch_size=BATCH_DATA)

test_data = tf.keras.preprocessing.image_dataset_from_directory(directory=test_dir,
                                                                image_size=IMG_SIZE,
                                                                label_mode="categorical",
                                                                batch_size=BATCH_DATA)


Found 750 files belonging to 10 classes.
Found 2500 files belonging to 10 classes.


In [8]:
train_data_10_percent

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 244, 244, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 10), dtype=tf.float32, name=None))>

In [9]:
# Check class names of our dataset 
train_data_10_percent.class_names

['chicken_curry',
 'chicken_wings',
 'fried_rice',
 'grilled_salmon',
 'hamburger',
 'ice_cream',
 'pizza',
 'ramen',
 'steak',
 'sushi']

In [10]:
# See an example of a batch 
for images , labels in train_data_10_percent.take(1) :
    print(images , labels)

tf.Tensor(
[[[[6.91056137e+01 5.09042435e+01 2.60558834e+01]
   [8.02306671e+01 5.69741325e+01 3.04229031e+01]
   [9.46269226e+01 6.40975266e+01 3.57929115e+01]
   ...
   [1.39547165e+02 8.15471725e+01 3.35471725e+01]
   [1.36647583e+02 7.86475830e+01 3.06475830e+01]
   [1.36089600e+02 7.80896072e+01 3.00896072e+01]]

  [[8.31598358e+01 5.58004570e+01 2.84258423e+01]
   [9.14622192e+01 6.09592857e+01 3.30105133e+01]
   [9.65818481e+01 6.07192650e+01 3.17662773e+01]
   ...
   [1.38745941e+02 8.07459412e+01 3.27459412e+01]
   [1.36261429e+02 7.82614212e+01 3.02614212e+01]
   [1.37047119e+02 7.90471191e+01 3.10471172e+01]]

  [[9.26140289e+01 5.56536903e+01 2.66598358e+01]
   [9.86265717e+01 5.96598358e+01 2.86761131e+01]
   [9.82494125e+01 5.72679558e+01 2.52848358e+01]
   ...
   [1.40228653e+02 8.22286530e+01 3.42286530e+01]
   [1.36000000e+02 7.80000000e+01 3.00000000e+01]
   [1.36904984e+02 7.89049911e+01 3.09049873e+01]]

  ...

  [[7.25580750e+01 6.05580788e+01 4.45580788e+01]
   [7

## Model 0 : Building a transfer learning model using the Keras Functional API

The sequential API is straight-forward , it runs our layers in sequential order .

In [12]:
# Create the base model with tf.keras.applications 

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False
)

# Freez the base model (so the underlying pre-trained patterns aren't updated during training)
base_model.trainable = False

# Create the inputs into our model
inputs = tf.keras.layers.Input(shape=(244, 244, 3) , name="input_layer")

# Normalize the inputs 
# x = tf.keras.layers.experimental.preprocessing.Rescaling(1/255.)(inputs)

# Pass the inputs to the base model 
x = base_model(inputs)

print(f"Shape after passing inputs through base model : {x.shape}")

# Average pool the outputs of the base model (aggregate the most importantt features)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)
print(f"Shape after global average pooling 2D : {x.shape}")

# Create the output activation layer 
outputs = tf.keras.layers.Dense(10 , activation="softmax" , name="output_layer")(x)

# Combine the inputs with the outputs into a model
model_0 = tf.keras.Model(inputs , outputs)

# Compile the model
model_0.compile(loss="categorical_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])   

# Fit the model 
history_10_percent = model_0.fit(train_data_10_percent,
                                  epochs=5,
                                  steps_per_epoch=len(train_data_10_percent),
                                  validation_data=test_data,
                                  validation_steps=len(test_data),
                                  callbacks=[create_tensorboard_callback(dir_name="transfer_learning",
                                  experiment_name="10_percent_data")])

Shape after passing inputs through base model : (None, 7, 7, 1280)
Shape after global average pooling 2D : (None, 1280)
Saving TensorBoard log files to: transfer_learning/10_percent_data/20260225-210441
Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 0.2401 - loss: 2.1698 - val_accuracy: 0.7112 - val_loss: 1.3502
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 276ms/step - accuracy: 0.7438 - loss: 1.2051 - val_accuracy: 0.8116 - val_loss: 0.9014
Epoch 3/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 232ms/step - accuracy: 0.8520 - loss: 0.8162 - val_accuracy: 0.8448 - val_loss: 0.7101
Epoch 4/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 12s 301ms/step - accuracy: 0.8317 - loss: 0.6954 - val_accuracy: 0.8588 - val_loss: 0.6178
Epoch 5/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 265ms/step - accuracy: 0.8856 - loss: 0.5580 - val_accuracy: 0.8668 - val_loss: 0.5583


In [13]:
model_0.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 244, 244, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling_layer    │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,088,003 (15.59 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 25,622 (100.09 KB)